In [1]:
# 直接拆解数据，使用最小二乘法对模型进行参数辨识
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
import math, datetime


In [9]:
data = pd.read_csv("../../../src/physical_verify/static/data/real_static_state/real(effective)_StaticPoint_6group_2025-02-21_08-55-07_2025-08-01_19-34-21.csv")
# 实验数据
P1_array = data['P1 (kPa)'].values
P2_array = data['P2 (kPa)'].values
theta1_array = data['theta1 (deg)'].values
theta2_array = data['theta2 (deg)'].values
# 实验数据
theta1 = torch.tensor(theta1_array, dtype=torch.float32)*torch.pi/180  # 输入 theta1
theta2 = torch.tensor(theta2_array, dtype=torch.float32)*torch.pi/180  # 输入 theta2
P1_actual = torch.tensor(P1_array, dtype=torch.float32)*1000  # 实验输出 P
P2_actual = torch.tensor(P2_array, dtype=torch.float32)*1000  # 实验输出 P
P2_actual.shape

torch.Size([13])

In [10]:
# make the data from the geometry model
# 为了简单, make data 不用 math
def get_geom_data(theta_1, theta_2):
    '''
        return: gLinvY, l1, l2 (各个分量)
    '''
    # 固定参数
    a_1, a_2, b_1, b_2, d_1, d_2 = 0.25, 0.25, 0.21213, 0.1, 0.06, 0.10
    beta_1, beta_2 = 8.13 / 180 * math.pi, 30 / 180 * math.pi
    g = 9.8

    A_x_O = d_1
    A_y_O = 0

    B_x_O = -d_2
    B_y_O = 0

    C_x_O = b_1 * math.cos(theta_1 - beta_1)
    C_y_O = b_1 * math.sin(theta_1 - beta_1)

    D_x_O = a_1 * math.cos(theta_1) + b_2 * math.cos(theta_1 + theta_2 + beta_2)
    D_y_O = a_1 * math.sin(theta_1) + b_2 * math.sin(theta_1 + theta_2 + beta_2)

    E_x_O = a_1 * math.cos(theta_1)
    E_y_O = a_1 * math.sin(theta_1)

    F_x_O = a_1 * math.cos(theta_1) + a_2 * math.cos(theta_1 + theta_2)
    F_y_O = a_1 * math.sin(theta_1) + a_2 * math.sin(theta_1 + theta_2)

    # 计算偏导数
    # 对 theta_1 的偏导数

    dA_x_O_dtheta_1 = 0
    dA_y_O_dtheta_1 = 0

    dB_x_O_dtheta_1 = 0
    dB_y_O_dtheta_1 = 0

    dC_x_O_dtheta_1 = -b_1 * math.sin(theta_1 - beta_1)
    dC_y_O_dtheta_1 = b_1 * math.cos(theta_1 - beta_1)

    dD_x_O_dtheta_1 = -a_1 * math.sin(theta_1) - b_2 * math.sin(theta_1 + theta_2 + beta_2)
    dD_y_O_dtheta_1 = a_1 * math.cos(theta_1) + b_2 * math.cos(theta_1 + theta_2 + beta_2)

    dE_x_O_dtheta_1 = -a_1 * math.sin(theta_1)
    dE_y_O_dtheta_1 = a_1 * math.cos(theta_1)

    dF_x_O_dtheta_1 = -a_1 * math.sin(theta_1) - a_2 * math.sin(theta_1 + theta_2)
    dF_y_O_dtheta_1 = a_1 * math.cos(theta_1) + a_2 * math.cos(theta_1 + theta_2)

    # 对 theta_2 的偏导数

    dA_x_O_dtheta_2 = 0.0
    dA_y_O_dtheta_2 = 0.0

    dB_x_O_dtheta_2 = 0.0
    dB_y_O_dtheta_2 = 0.0

    dC_x_O_dtheta_2 = 0.0
    dC_y_O_dtheta_2 = 0.0

    dD_x_O_dtheta_2 = -b_2 * math.sin(theta_1 + theta_2 + beta_2)
    dD_y_O_dtheta_2 = b_2 * math.cos(theta_1 + theta_2 + beta_2)

    dE_x_O_dtheta_2 = 0.0
    dE_y_O_dtheta_2 = 0.0

    dF_x_O_dtheta_2 = -a_2 * math.sin(theta_1 + theta_2)
    dF_y_O_dtheta_2 = a_2 * math.cos(theta_1 + theta_2)

    # 计算长度 l1 和 l2
    l_1 = math.sqrt((A_x_O - C_x_O)**2 + (A_y_O - C_y_O)**2)
    l_2 = math.sqrt((B_x_O - D_x_O)**2 + (B_y_O - D_y_O)**2)

    # 计算偏导数
    # 偏导数 d/dtheta_1
    dl_1_dtheta_1 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_1 - dC_x_O_dtheta_1) + (A_y_O - C_y_O) * (dA_y_O_dtheta_1 - dC_y_O_dtheta_1))
    dl_2_dtheta_1 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_1 - dD_x_O_dtheta_1) + (B_y_O - D_y_O) * (dB_y_O_dtheta_1 - dD_y_O_dtheta_1))

    # 偏导数 d/dtheta_2
    dl_1_dtheta_2 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_2 - dC_x_O_dtheta_2) + (A_y_O - C_y_O) * (dA_y_O_dtheta_2 - dC_y_O_dtheta_2))
    dl_2_dtheta_2 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_2 - dD_x_O_dtheta_2) + (B_y_O - D_y_O) * (dB_y_O_dtheta_2 - dD_y_O_dtheta_2))

    # print(dl_1_dtheta_2)  # check the model

    # 等式右侧
    # print(type(dE_y_O_dtheta_2/2))
    # print(type((dE_y_O_dtheta_2+dF_y_O_dtheta_2)/2))
    # print(type(dC_y_O_dtheta_2/2))
    # print(type(dD_y_O_dtheta_2/2))
    
    # _ = math.stack([(dE_y_O_dtheta_2/2), ((dE_y_O_dtheta_2+dF_y_O_dtheta_2)/2), (dC_y_O_dtheta_2/2), (dD_y_O_dtheta_2/2)], dim=1)
    Y_mat = np.array([
        [(dE_y_O_dtheta_1/2), ((dE_y_O_dtheta_1+dF_y_O_dtheta_1)/2), (dC_y_O_dtheta_1/2), (dD_y_O_dtheta_1/2)], 
        [(dE_y_O_dtheta_2/2), ((dE_y_O_dtheta_2+dF_y_O_dtheta_2)/2), (dC_y_O_dtheta_2/2), (dD_y_O_dtheta_2/2)]])

    # 等式左侧
    L_mat = np.array([
        [dl_1_dtheta_1, dl_2_dtheta_1],
        [dl_1_dtheta_2, dl_2_dtheta_2]])

    return Y_mat, L_mat, l_1, l_2



In [ ]:
# get_geom_data(1,1)

# g = 9.8
# m1, m2, m3, m4 = 0.086, 0.1033, 186.48 * 1e-3, 272.66 * 1e-3
# # k1, k2 = 300, 300
# # l10, l20 = 0.174, 0.252
# Y_mat, L_mat, l_1, l_2 = get_geom_data(1, 1)
# print(Y_mat, L_mat, l_1, l_2)

# 模拟计算
# for t1, t2 in zip(theta1, theta2):
#     # print(t1, t2)
#     Y_mat, L_mat, l_1, l_2 = get_geom_data(t1, t2)
#     F = -g*np.linalg.inv(L_mat)@Y_mat@(np.array([m1, m2, m3, m4]).reshape(-1, 1)) + np.array([k1*(l_1-l10), k2*(l_2-l20)]).reshape(-1, 1)
#     # print(F[0], F[1])


In [11]:
# 构造最小二乘问题（补变量Trick）——气压模型
# 变量集：S1, S2, m1, m2, m3, m4, k1, k2, -k1*l10, -k2*l20
var_num = 6
data_size = len(theta1)
A_mat = np.zeros((data_size*2, var_num))
b_mat = np.zeros((data_size*2, 1))
m1, m2, m3, m4 = 0.086, 0.1033, 186.48 * 1e-3, 272.66 * 1e-3
g = 9.8
m_array = np.array([m1, m2, m3, m4]).reshape(-1, 1)

for i in range(data_size):      # 第 i 大组
    t1, t2 = theta1[i], theta2[i]
    Y_mat, L_mat, l_1, l_2 = get_geom_data(t1, t2)
    # print(Y_mat, L_mat, l_1, l_2)
    temp_mat = -g*np.linalg.inv(L_mat)@Y_mat@m_array
    A_mat[i*2, 0] = temp_mat[0, 0]
    A_mat[i*2, 1] = 0
    # A_mat[i*2, 2] = temp_mat[0, 0]
    # A_mat[i*2, 3] = temp_mat[0, 1]
    # A_mat[i*2, 4] = temp_mat[0, 2]
    # A_mat[i*2, 5] = temp_mat[0, 3]
    A_mat[i*2, 2] = l_1
    A_mat[i*2, 3] = 0
    A_mat[i*2, 4] = 1
    A_mat[i*2, 5] = 0

    A_mat[i*2+1, 0] = 0
    A_mat[i*2+1, 1] = temp_mat[1, 0]
    # A_mat[i*2+1, 2] = temp_mat[1, 0]
    # A_mat[i*2+1, 3] = temp_mat[1, 1]
    # A_mat[i*2+1, 4] = temp_mat[1, 2]
    # A_mat[i*2+1, 5] = temp_mat[1, 3]
    A_mat[i*2+1, 2] = 0
    A_mat[i*2+1, 3] = l_2
    A_mat[i*2+1, 4] = 0
    A_mat[i*2+1, 5] = 1

    b_mat[i*2, 0] = P1_actual[i]
    b_mat[i*2+1, 0] = P2_actual[i]
    

# sove Ax = 0
solution = np.linalg.inv(A_mat.T@A_mat)@A_mat.T@b_mat
s1_prime = 1/solution[0, 0]
s2_prime = 1/solution[1, 0]
k1_prime = solution[2, 0]*s1_prime
k2_prime = solution[3, 0]*s2_prime
l10_prime = -solution[4, 0]/k1_prime*s1_prime
l20_prime = -solution[5, 0]/k2_prime*s2_prime

# print the result
print(f"s1: {s1_prime}, s2: {s2_prime}, \nk1: {k1_prime}, k2: {k2_prime}, \nl10: {l10_prime}, l20: {l20_prime}")

x_mat = A_mat@solution
x_mat = x_mat.reshape(-1, 2)
p_mat = np.stack((P1_actual, P2_actual), axis=1)

# print(x_mat)
# print(p_mat)
print(f"PressureError(sqrtMSE): {math.sqrt(np.linalg.norm(x_mat-p_mat)**2/2/data_size)/1000:.6f}")

# 力大小
F_mat = p_mat
F_mat[:, 0] = F_mat[:, 0]*s1_prime
F_mat[:, 1] = F_mat[:, 1]*s2_prime
print(F_mat)
print(theta1[2]-(-0.295+math.pi/2), (theta2[2]-0.569))

theta_mat = np.stack([theta1, theta2], axis=1)
theta_mat

s1: 0.0002622743904575065, s2: 0.0004647463729979666, 
k1: 39.960784254451305, k2: 160.1731540852416, 
l10: 0.08205225743609154, l20: 0.2642717186495769
PressureError(sqrtMSE): 0.805867
[[ 0.         0.       ]
 [ 0.         4.647464 ]
 [ 0.         9.294928 ]
 [ 0.        13.942391 ]
 [ 2.6227438  0.       ]
 [ 2.6227438  4.647464 ]
 [ 2.6227438  9.294928 ]
 [ 5.2454877  0.       ]
 [ 5.2454877  4.647464 ]
 [ 7.868232   0.       ]
 [ 7.868232   4.647464 ]
 [10.490975   0.       ]
 [13.11372    0.       ]]
tensor(0.0071) tensor(-0.0530)


array([[1.2755774 , 1.0714227 ],
       [1.3015182 , 0.79277396],
       [1.2829165 , 0.5160223 ],
       [1.2929783 , 0.21563716],
       [1.4700559 , 0.86282104],
       [1.4716564 , 0.5444171 ],
       [1.4977683 , 0.19920665],
       [1.6180999 , 0.6955888 ],
       [1.6336073 , 0.34381416],
       [1.7746457 , 0.5192285 ],
       [1.7917794 , 0.18421601],
       [1.9147938 , 0.37423176],
       [2.0679536 , 0.17999405]], dtype=float32)

In [102]:
# load npy
# real vs sim(ideal)
print("Real vs Sim(ideal)")
# tmp = np.load("/Users/flypig/Documents/Coding/MujocoLearn/data/Exp-sim-real constrast-20250219_230253/StaticState_list.npy")        # the ideal model
tmp = np.load("/Users/flypig/Documents/Coding/MujocoLearn/data/Exp-sim-real constrast-20250219_231642/StaticState_list.npy")        # the geom model

theta_sim = tmp[:,3:5]
theta_sim[:, 0] = theta_sim[:, 0] + math.pi/2

# for geom model
theta_sim[:, 1] = theta_sim[:, 1] + math.pi/2

theta_error = (theta_sim - theta_mat)
print(theta_error)
print(f"Error_relative: {theta_error/theta_sim}")
# data_mat for LLM to form a table
data_mat = np.hstack((P1_actual.reshape(data_size, 1)/1000, P2_actual.reshape(data_size, 1)/1000, (theta_sim)*180/math.pi, (theta_mat)*180/math.pi, theta_error*180/math.pi, theta_error/theta_real))
print(data_mat)

# save
dataShow = pd.DataFrame(data_mat, columns=['Pressure1/kPa', 'Pressure2/kPa', 'Theta1_real/deg', 'Theta2_real/deg', 'Theta1_sim/deg', 'Theta2_sim/deg', 'Theta1_error/deg', 'Theta2_error/deg', 'Theta1_errorRelative', 'Theta2_errorRelative'])
dataShow.to_csv('../../log/matrix_data.csv', index=False)

# in Rad
print(f"Max error: {np.max(np.abs(theta_error))}")
print(f"SqrtMSE: {math.sqrt(np.sum(theta_error**2)/theta_error.size)}")

Real vs Sim(ideal)
[[ 0.00120894 -0.02078929]
 [ 0.00413228 -0.01785462]
 [ 0.01240105  0.00708697]
 [ 0.00965819 -0.07370157]
 [-0.01784607  0.01286322]
 [-0.00437564 -0.01100087]
 [-0.00397715  0.0817909 ]
 [-0.01833559  0.02005382]
 [ 0.00348028 -0.04051692]
 [ 0.0265297  -0.02491953]
 [ 0.00878614 -0.08448486]
 [ 0.00323738  0.00387313]
 [ 0.00988819 -0.02572056]]
Error_relative: [[ 0.00093652 -0.01950498]
 [ 0.00318806 -0.02255775]
 [ 0.00949771  0.01412162]
 [ 0.00738135 -0.36723251]
 [-0.01232537  0.01414062]
 [-0.00300161 -0.01779225]
 [-0.00270669  0.26801755]
 [-0.01145716  0.02678657]
 [ 0.00214233 -0.09607782]
 [ 0.01508196 -0.04374527]
 [ 0.00491628 -0.42281904]
 [ 0.00168205  0.01054797]
 [ 0.00477061 -0.17744167]]
[[ 0.00000000e+00  0.00000000e+00  7.39619350e+01  6.10684332e+01
   7.38926620e+01  6.22595673e+01  6.92670624e-02 -1.19113851e+00
   9.36523123e-04  4.11708952e-02]
 [ 0.00000000e+00  1.00000000e+01  7.42654192e+01  4.53500184e+01
   7.40286560e+01  4.6373012

In [26]:
# 构造最小二乘问题（补变量Trick）——压力模型，最小化力的MSE，得到模型之后再计算气压的MSE。
# 变量集：S1, S2, k1, k2, -k1*l10, -k2*l20
var_num = 6
data_size = len(theta1)
A_mat = np.zeros((data_size*2, var_num))
b_mat = np.zeros((data_size*2, 1))
m1, m2, m3, m4 = 0.086, 0.1033, 186.48 * 1e-3, 272.66 * 1e-3
g = 9.8
m_array = np.array([m1, m2, m3, m4]).reshape(-1, 1)

for i in range(data_size):      # 第 i 大组
    t1, t2 = theta1[i], theta2[i]
    Y_mat, L_mat, l_1, l_2 = get_geom_data(t1, t2)
    # print(Y_mat, L_mat, l_1, l_2)
    temp_mat = g*np.linalg.inv(L_mat)@Y_mat@m_array
    A_mat[i*2, 0] = -P1_actual[i]
    A_mat[i*2, 1] = 0
    # A_mat[i*2, 2] = temp_mat[0, 0]
    # A_mat[i*2, 3] = temp_mat[0, 1]
    # A_mat[i*2, 4] = temp_mat[0, 2]
    # A_mat[i*2, 5] = temp_mat[0, 3]
    A_mat[i*2, 2] = l_1
    A_mat[i*2, 3] = 0
    A_mat[i*2, 4] = 1
    A_mat[i*2, 5] = 0

    A_mat[i*2+1, 0] = 0
    A_mat[i*2+1, 1] = -P2_actual[i]
    # A_mat[i*2+1, 2] = temp_mat[1, 0]
    # A_mat[i*2+1, 3] = temp_mat[1, 1]
    # A_mat[i*2+1, 4] = temp_mat[1, 2]
    # A_mat[i*2+1, 5] = temp_mat[1, 3]
    A_mat[i*2+1, 2] = 0
    A_mat[i*2+1, 3] = l_2
    A_mat[i*2+1, 4] = 0
    A_mat[i*2+1, 5] = 1

    b_mat[i*2, 0] = temp_mat[0, 0]
    b_mat[i*2+1, 0] = temp_mat[1, 0]
    
# solve Ax = b
solution = np.linalg.inv(A_mat.T@A_mat)@A_mat.T@b_mat
s1_prime = solution[0, 0]
s2_prime = solution[1, 0]
k1_prime = solution[2, 0]
k2_prime = solution[3, 0]
l10_prime = -solution[4, 0]/k1_prime
l20_prime = -solution[5, 0]/k2_prime

# print the result
print(f"s1: {s1_prime}, s2: {s2_prime}, \nk1: {k1_prime}, k2: {k2_prime}, \nl10: {l10_prime}, l20: {l20_prime}")

x_mat = A_mat@solution
print(x_mat - b_mat)
print(x_mat)
x_mat = x_mat.reshape(-1, 2)
x_mat[:,0] = x_mat[:,0]/s1_prime
x_mat[:,1] = x_mat[:,1]/s2_prime
p_mat = np.stack((P1_actual, P2_actual), axis=1)

print(x_mat)
print(p_mat)
print(math.sqrt(np.linalg.norm(x_mat-p_mat)**2/2/data_size)/1000)

s1: 0.00018027918560369811, s2: 7.437764862025753e-05, 
k1: -46.68707623412483, k2: 8.950055506603391, 
l10: 0.2938893588532079, l20: 0.0037424213578018777
[[ 0.16048825]
 [ 0.20676436]
 [ 0.08550936]
 [ 0.06154466]
 [-0.117006  ]
 [ 0.08042577]
 [-0.12401361]
 [-0.03632713]
 [ 0.28661822]
 [ 0.21446369]
 [ 0.04954805]
 [-0.00896834]
 [-0.11537912]
 [ 0.17676148]
 [ 0.16132869]
 [ 0.16550999]
 [-0.11574324]
 [-0.13801757]
 [-0.34816171]
 [ 0.09871191]
 [-0.24919841]
 [-0.31995186]
 [ 0.14992532]
 [-0.11246484]
 [ 0.1760842 ]
 [-0.38845211]]
[[ 4.60200231]
 [ 2.4172819 ]
 [ 4.59541682]
 [ 1.91738863]
 [ 4.5919664 ]
 [ 1.43298073]
 [ 4.57665837]
 [ 0.84586246]
 [ 2.30718912]
 [ 2.4319581 ]
 [ 2.31731971]
 [ 1.92148415]
 [ 2.28593534]
 [ 1.4736159 ]
 [ 0.07916473]
 [ 2.44190482]
 [ 0.07267911]
 [ 1.9135372 ]
 [-2.03307322]
 [ 2.45256849]
 [-2.15520747]
 [ 1.90404886]
 [-4.32677876]
 [ 2.46637507]
 [-6.47186402]
 [ 2.47834267]]
[[ 25527.08621649  32500.11183269]
 [ 25490.55679969  25779.09